# Nettoyage des données & Création du label Churn

**Projet BI 25/26 — Groupe 7**  
Analyse des ventes et tendances e-commerce


### Logique du Churn
Un client est considéré **churné** si :
- Sa **dernière visite** sur le site date de plus de **90 jours** ET
- Son **dernier achat** date de plus de **90 jours**


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)

CHURN_DAYS = 270  # seuil en jours

## 1. Chargement des données

In [15]:
customers = pd.read_csv('dataset/Customers_data.csv')
traffic   = pd.read_csv('dataset/Web_traffic.csv')

customers.columns = customers.columns.str.lower().str.strip().str.replace(' ', '_')
traffic.columns   = traffic.columns.str.lower().str.strip().str.replace(' ', '_')

print(f'customers : {customers.shape}')
print(f'traffic   : {traffic.shape}')
customers.head(3)

customers : (250000, 13)
traffic   : (300000, 10)


,customer_id,purchase_date,product_category,product_price,quantity,total_purchase_amount,payment_method,customer_age,returns,customer_name,age,gender,churn
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,31,1.0,John Rivera,31,Female,0
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,31,1.0,John Rivera,31,Female,0
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,31,1.0,John Rivera,31,Female,0


In [16]:
traffic.head(3)

,session_id,customer_id,date,traffic_source,device,session_duration_sec,pages_viewed,purchased,cart_abandoned,added_to_cart
0,SES104941,43948.0,2023-04-11,Direct,Desktop,721.537446,12.0,Yes,No,Yes
1,SES151775,40980.0,2022-03-16,Email Marketing,Mobile,471.922093,8.0,Yes,No,Yes
2,SES215253,NaN,2023-08-02,Referral,Mobile,89.155834,5.0,No,Yes,Yes


## 2. Nettoyage — Customers

In [17]:
customers.columns = customers.columns.str.lower()

# Valeurs manquantes
print('Valeurs manquantes avant nettoyage :')
print(customers.isnull().sum())

#remplacer les manquants de returnS par le mode
before = customers['returns'].isnull().sum()
customers['returns'] = customers['returns'].fillna(customers['returns'].mode()[0])
print(f'\nReturns : {before} manquants imputés par le mode ({customers["returns"].mode()[0]})')

# Doublons
dupes = customers.duplicated().sum()
customers = customers.drop_duplicates()
print(f'Doublons supprimés : {dupes}')

print(f'\nCustomers nettoyé : {customers.shape}')

Valeurs manquantes avant nettoyage :
customer_id                  0
purchase_date                0
product_category             0
product_price                0
quantity                     0
total_purchase_amount        0
payment_method               0
customer_age                 0
returns                  47382
customer_name                0
age                          0
gender                       0
churn                        0
dtype: int64

Returns : 47382 manquants imputés par le mode (1.0)
Doublons supprimés : 0

Customers nettoyé : (250000, 13)


In [18]:
# Renommons les colonnes
customers = customers.rename(columns={
    'Customer ID'          : 'customer_id',
    'Purchase Date'        : 'purchase_date',
    'Product Category'     : 'product_category',
    'Product Price'        : 'product_price',
    'Quantity'             : 'quantity',
    'Total Purchase Amount': 'total_purchase_amount',
    'Payment Method'       : 'payment_method',
    'Customer Age'         : 'customer_age',
    'Returns'              : 'returns',
    'Gender'               : 'gender',
})

# Supprimer la colonne churn
if 'Churn' in customers.columns:
    customers = customers.drop(columns=['Churn'])
    print('Colonne Churn originale supprimée — sera recalculée depuis les activités')

# Supprimer customer_age
if 'customer_age' in customers.columns:
    customers = customers.drop(columns=['customer_age'])

# Conversion de la date
customers['purchase_date'] = pd.to_datetime(customers['purchase_date'])

print(customers.dtypes)

customer_id                       int64
purchase_date            datetime64[us]
product_category                    str
product_price                     int64
quantity                          int64
total_purchase_amount             int64
payment_method                      str
returns                         float64
customer_name                       str
age                               int64
gender                              str
churn                             int64
dtype: object


## 3. Nettoyage — Traffic Web

In [19]:
# Renommage
traffic = traffic.rename(columns={
    'Session_ID'           : 'session_id',
    'Customer_ID'          : 'customer_id',
    'Date'                 : 'date',
    'Traffic_Source'       : 'traffic_source',
    'Device'               : 'device',
    'Session_Duration_Sec' : 'session_duration_sec',
    'Pages_Viewed'         : 'pages_viewed',
    'Purchased'            : 'purchased',
    'Cart_Abandoned'       : 'cart_abandoned',
    'Added_To_Cart'        : 'added_to_cart'
})

# Suppression lignes sans customer_id
before = len(traffic)
traffic = traffic.dropna(subset=['customer_id'])
traffic['customer_id'] = traffic['customer_id'].astype(int)
print(f'Lignes supprimées (customer_id manquant) : {before - len(traffic)}')

# Conversion date
traffic['date'] = pd.to_datetime(traffic['date'])

# Encodage binaire
for col in ['purchased', 'cart_abandoned', 'added_to_cart']:
    traffic[col] = traffic[col].map({'Yes': 1, 'No': 0})

# Capper les outliers de durée de session au percentile 99
p99 = traffic['session_duration_sec'].quantile(0.99)
outliers = (traffic['session_duration_sec'] > p99).sum()
traffic['session_duration_sec'] = traffic['session_duration_sec'].clip(upper=p99)
print(f'Outliers session_duration cappés au p99 ({p99:.1f} sec) : {outliers} lignes')

# Doublons
dupes = traffic.duplicated().sum()
traffic = traffic.drop_duplicates()
print(f'Doublons supprimés : {dupes}')

print(f'\nTraffic nettoyé : {traffic.shape}')

Lignes supprimées (customer_id manquant) : 84087
Outliers session_duration cappés au p99 (789.6 sec) : 2005 lignes
Doublons supprimés : 0

Traffic nettoyé : (215913, 10)


## 4. Création du label Churn basé sur les activités

> **Règle :** Un client est churné si sa dernière visite ET son dernier achat datent tous les deux de plus de 90 jours.

In [20]:
# Date de référence = date la plus récente dans les données
reference_date = max(traffic['date'].max(), customers['purchase_date'].max())
print(f'Date de référence : {reference_date.date()}')

# Dernière visite par client (depuis traffic) 

last_visit = traffic.groupby('customer_id')['date'].max().reset_index()
last_visit.columns = ['customer_id', 'last_visit_date']
last_visit['jours_depuis_visite'] = (reference_date - last_visit['last_visit_date']).dt.days

#Dernier achat par client (depuis customers)
last_purchase = customers.groupby('customer_id')['purchase_date'].max().reset_index()
last_purchase.columns = ['customer_id', 'last_purchase_date']
last_purchase['jours_depuis_achat'] = (reference_date - last_purchase['last_purchase_date']).dt.days

#Fusion
churn_labels = last_visit.merge(last_purchase, on='customer_id', how='outer')

# Clients sans visite = inactifs depuis très longtemps
churn_labels['jours_depuis_visite']  = churn_labels['jours_depuis_visite'].fillna(9999)
churn_labels['jours_depuis_achat']   = churn_labels['jours_depuis_achat'].fillna(9999)

# Règle churn : inactif ET pas d'achat depuis 90 jours
churn_labels['churn'] = (
    (churn_labels['jours_depuis_visite'] > CHURN_DAYS) &
    (churn_labels['jours_depuis_achat']  > CHURN_DAYS)
).astype(int)

print(f'\nClients analysés     : {len(churn_labels)}')
print(f'Clients Churné (1)   : {churn_labels["churn"].sum()} ({churn_labels["churn"].mean():.1%})')
print(f'Clients Actifs (0)   : {(churn_labels["churn"]==0).sum()} ({1-churn_labels["churn"].mean():.1%})')
churn_labels.head()

Date de référence : 2023-09-13

Clients analysés     : 49661
Clients Churné (1)   : 9492 (19.1%)
Clients Actifs (0)   : 40169 (80.9%)


,customer_id,last_visit_date,jours_depuis_visite,last_purchase_date,jours_depuis_achat,churn
0,1,2022-12-22,265.0,2022-11-29 06:48:25,288,0
1,2,2022-09-04,374.0,2023-07-03 17:26:19,72,0
2,3,2023-02-26,199.0,2023-02-03 03:58:07,222,0
3,4,2022-06-29,441.0,2022-06-29 03:41:09,441,1
4,5,2023-07-12,63.0,2022-07-16 04:08:09,424,0


## 5. Construction des datasets finaux

In [21]:
def safe_mode(x):
    m = x.mode()
    return m.iloc[0] if len(m) > 0 else np.nan

# Agrégation traffic par client
traffic_agg = traffic.groupby('customer_id').agg(
    nb_sessions          = ('session_id',           'count'),
    total_duration_sec   = ('session_duration_sec', 'sum'),
    avg_duration_sec     = ('session_duration_sec', 'mean'),
    avg_pages_viewed     = ('pages_viewed',          'mean'),
    nb_purchases_traffic = ('purchased',             'sum'),
    nb_cart_abandoned    = ('cart_abandoned',        'sum'),
    nb_added_to_cart     = ('added_to_cart',         'sum'),
    taux_achat_session   = ('purchased',             'mean'),
    taux_abandon_session = ('cart_abandoned',        'mean'),
    main_traffic_source  = ('traffic_source',        safe_mode),
    main_device          = ('device',                safe_mode)
).reset_index()

# Agrégation customers par client
customers_agg = customers.groupby('customer_id').agg(
    gender              = ('gender',                'first'),
    returns             = ('returns',               'mean'),
    avg_product_price   = ('product_price',         'mean'),
    avg_quantity        = ('quantity',              'mean'),
    avg_purchase_amount = ('total_purchase_amount', 'mean'),
    nb_transactions     = ('purchase_date',         'count'),
    payment_method      = ('payment_method',        safe_mode),
).reset_index()

# Dataset CHURN : une ligne par client
df_churn = customers_agg \
    .merge(traffic_agg, on='customer_id', how='left') \
    .merge(churn_labels[['customer_id', 'jours_depuis_visite', 'jours_depuis_achat', 'churn']], on='customer_id', how='left')

df_churn['churn'] = df_churn['churn'].fillna(1).astype(int)  # sans activité = churné

# Dataset PURCHASED : une ligne par session
customers_static = customers.groupby('customer_id').agg(
    gender              = ('gender',                'first'),
    returns             = ('returns',               'mean'),
    avg_product_price   = ('product_price',         'mean'),
    avg_purchase_amount = ('total_purchase_amount', 'mean'),
    nb_transactions     = ('purchase_date',         'count'),
    payment_method      = ('payment_method',        safe_mode),
).reset_index()

df_purchased = traffic.merge(customers_static, on='customer_id', how='left')

print(f'Dataset CHURN     : {df_churn.shape}   | Taux churn : {df_churn["churn"].mean():.1%}')
print(f'Dataset PURCHASED : {df_purchased.shape} | Taux achat : {df_purchased["purchased"].mean():.1%}')

print('\n--- df_churn (3 lignes) ---')
display(df_churn.head(3))
print('\n--- df_purchased (3 lignes) ---')
display(df_purchased.head(3))

Dataset CHURN     : (49661, 22)   | Taux churn : 19.1%
Dataset PURCHASED : (215913, 16) | Taux achat : 27.8%

--- df_churn (3 lignes) ---


,customer_id,gender,returns,avg_product_price,avg_quantity,avg_purchase_amount,nb_transactions,payment_method,nb_sessions,total_duration_sec,avg_duration_sec,avg_pages_viewed,nb_purchases_traffic,nb_cart_abandoned,nb_added_to_cart,taux_achat_session,taux_abandon_session,main_traffic_source,main_device,jours_depuis_visite,jours_depuis_achat,churn
0,1,Female,0.666667,373.333333,5.00,2096.666667,3,Credit Card,6.0,1498.651687,249.775281,4.000000,1.0,5.0,3.0,0.166667,0.833333,Direct,Mobile,265.0,288,0
1,2,Female,0.666667,338.333333,3.00,2746.833333,6,PayPal,4.0,2099.764243,524.941061,6.000000,3.0,1.0,4.0,0.750000,0.250000,Organic Search,Desktop,374.0,72,0
2,3,Male,0.750000,222.750000,3.75,2355.750000,4,Credit Card,4.0,472.199929,157.399976,3.333333,0.0,4.0,1.0,0.000000,1.000000,Social Media,Mobile,199.0,222,0



--- df_purchased (3 lignes) ---


,session_id,customer_id,date,traffic_source,device,session_duration_sec,pages_viewed,purchased,cart_abandoned,added_to_cart,gender,returns,avg_product_price,avg_purchase_amount,nb_transactions,payment_method
0,SES104941,43948,2023-04-11,Direct,Desktop,721.537446,12.0,1,0,1,Male,0.60,275.2,3216.6,5,PayPal
1,SES151775,40980,2022-03-16,Email Marketing,Mobile,471.922093,8.0,1,0,1,Male,0.25,236.5,4357.5,4,Credit Card
2,SES399321,24511,2022-09-19,Social Media,Desktop,159.336440,2.0,0,1,0,Female,0.60,278.8,3229.8,5,Cash


## 6. Sauvegarde

In [22]:
os.makedirs('dataset/cleaned', exist_ok=True)

customers.to_csv('dataset/cleaned/customers_clean.csv',   index=False)
traffic.to_csv('dataset/cleaned/traffic_clean.csv',       index=False)
df_churn.to_csv('dataset/cleaned/df_churn.csv',           index=False)
df_purchased.to_csv('dataset/cleaned/df_purchased.csv',   index=False)
churn_labels.to_csv('dataset/cleaned/churn_labels.csv',   index=False)

print('Fichiers sauvegardés dans dataset/cleaned/ :')
for f in os.listdir('dataset/cleaned'):
    size = os.path.getsize(f'dataset/cleaned/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')

Fichiers sauvegardés dans dataset/cleaned/ :
  churn_labels.csv  (2379.6 KB)
  clients_risque_churn.csv  (2473.3 KB)
  customers_clean.csv  (20194.0 KB)
  df_churn.csv  (8196.6 KB)
  df_purchased.csv  (25416.0 KB)
  traffic_clean.csv  (15465.3 KB)
